# 04 — Evaluation & Threshold Calibration

In [ ]:
import os, sys, json
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/anomaly_detection'
sys.path.insert(0, os.path.join(BASE_DIR, 'project_src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from training.config import ProjectConfig, set_seed, get_device
from training.model.scratch_3dcnn import Scratch3DCNN
from training.dataset.transforms import VideoTransform, decode_video_clips
from training.dataset.mil_dataset import UCFCrimeManifest, MILBagValidationDataset, val_collate_fn
from training.metrics import (
    compute_video_level_auc, compute_frame_level_auc,
    compute_false_alarm_rate, calibrate_threshold,
    compute_category_breakdown, plot_roc_pr_curves
)

config = ProjectConfig()
set_seed(config.SEED)
device = get_device()
print(f'Device: {device}')

## 1. Load Best Checkpoint

In [ ]:
model = Scratch3DCNN().to(device)

best_ckpt_path = os.path.join(config.CHECKPOINT_DIR, f'best_model_seed{config.SEED}.pt')
assert os.path.exists(best_ckpt_path), f'Best checkpoint not found: {best_ckpt_path}'

checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded best checkpoint from epoch {checkpoint.get("epoch", "?")}')
print(f'Validation AUC at save: {checkpoint.get("val_auc", "?")}')

## 2. Score All Test Videos

In [ ]:
test_manifest = UCFCrimeManifest(os.path.join(BASE_DIR, 'manifest_test.csv'))
test_transform = VideoTransform(mode='test')

test_dataset = MILBagValidationDataset(
    manifest=test_manifest,
    transform=test_transform,
    clip_length=config.CLIP_LENGTH,
    clip_stride=config.CLIP_STRIDE,
    target_fps=config.TARGET_FPS,
)

video_max_scores = []
video_labels = []
video_names = []
video_categories = []
all_clip_scores = []
all_clip_labels = []

with torch.no_grad():
    for i in tqdm(range(len(test_dataset)), desc='Scoring test videos'):
        clips, label, video_path, timestamps = test_dataset[i]
        clips = clips.to(device)
        
        # Score clips in batches to avoid OOM
        batch_size = 32
        scores = []
        for j in range(0, len(clips), batch_size):
            batch = clips[j:j+batch_size]
            logits = model(batch)
            batch_scores = torch.sigmoid(logits).squeeze(-1)
            scores.append(batch_scores)
        
        scores = torch.cat(scores).cpu().numpy()
        max_score = float(np.max(scores))
        
        video_max_scores.append(max_score)
        video_labels.append(label)
        video_names.append(os.path.basename(video_path))
        video_categories.append(test_manifest.get_category(video_path))
        
        # For frame-level AUC, expand clip scores to frames
        # (each clip covers clip_length frames with clip_stride overlap)
        all_clip_scores.extend(scores.tolist())
        all_clip_labels.extend([label] * len(scores))

print(f'\nScored {len(video_max_scores)} test videos')

## 3. Compute Metrics

In [ ]:
# Video-level AUC
video_auc = compute_video_level_auc(video_max_scores, video_labels)
print(f'Video-level ROC-AUC: {video_auc:.4f}')

# Frame/clip-level metrics
clip_metrics = compute_frame_level_auc(
    np.array(all_clip_scores),
    np.array(all_clip_labels)
)
print(f'Clip-level ROC-AUC: {clip_metrics["roc_auc"]:.4f}')
print(f'Clip-level PR-AUC: {clip_metrics["pr_auc"]:.4f}')

# False alarm rate on normal videos
normal_indices = [i for i, l in enumerate(video_labels) if l == 0]
normal_scores_list = []  # We'd need per-video clip scores here
# For now, compute from video-level
print(f'\nNormal videos scored: {len(normal_indices)}')
normal_max_scores = [video_max_scores[i] for i in normal_indices]
print(f'Normal max score stats: mean={np.mean(normal_max_scores):.4f}, max={np.max(normal_max_scores):.4f}')

## 4. Threshold Calibration (on Validation Set)

In [ ]:
# Load validation scores
val_manifest = UCFCrimeManifest(os.path.join(BASE_DIR, 'manifest_val.csv'))
val_dataset_for_cal = MILBagValidationDataset(
    manifest=val_manifest,
    transform=VideoTransform(mode='val'),
    clip_length=config.CLIP_LENGTH,
    clip_stride=config.CLIP_STRIDE,
    target_fps=config.TARGET_FPS,
)

val_scores = []
val_labels_list = []

with torch.no_grad():
    for i in tqdm(range(len(val_dataset_for_cal)), desc='Scoring val videos'):
        clips, label, _, _ = val_dataset_for_cal[i]
        clips = clips.to(device)
        scores = []
        for j in range(0, len(clips), 32):
            batch = clips[j:j+32]
            logits = model(batch)
            batch_scores = torch.sigmoid(logits).squeeze(-1)
            scores.append(batch_scores)
        scores = torch.cat(scores).cpu().numpy()
        val_scores.append(float(np.max(scores)))
        val_labels_list.append(label)

# Calibrate threshold
threshold = calibrate_threshold(val_scores, val_labels_list, target_fpr=0.05)
print(f'\nCalibrated threshold (target FPR=0.05): {threshold:.4f}')

# Freeze threshold
threshold_info = {
    'threshold': threshold,
    'calibrated_on': 'validation_set',
    'target_fpr': 0.05,
    'seed': config.SEED,
}
threshold_path = os.path.join(config.CHECKPOINT_DIR, 'calibrated_threshold.json')
with open(threshold_path, 'w') as f:
    json.dump(threshold_info, f, indent=2)
print(f'Threshold frozen to: {threshold_path}')

## 5. Final Test Metrics (Frozen Threshold)

In [ ]:
# Apply threshold to test
predictions = [1 if s >= threshold else 0 for s in video_max_scores]

from sklearn.metrics import classification_report, confusion_matrix

print('Classification Report (video-level):')
print(classification_report(video_labels, predictions, target_names=['Normal', 'Anomalous']))
print('\nConfusion Matrix:')
print(confusion_matrix(video_labels, predictions))

## 6. Category Breakdown & Failure Analysis

In [ ]:
cat_df = compute_category_breakdown(
    video_max_scores, video_labels, video_categories, threshold
)
print('\nPer-Category Performance:')
print(cat_df.to_string(index=False))
cat_df.to_csv(os.path.join(config.RUNS_DIR, f'category_breakdown_seed{config.SEED}.csv'), index=False)

In [ ]:
import cv2

def compute_failure_proxies(video_path):
    """Compute automatic proxies for failure conditions."""
    cap = cv2.VideoCapture(video_path)
    brightnesses = []
    blurs = []
    motions = []
    prev_gray = None
    
    frame_count = 0
    while frame_count < 100:  # sample first 100 frames
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Mean brightness
        brightnesses.append(np.mean(gray))
        
        # Laplacian variance (blur)
        blurs.append(cv2.Laplacian(gray, cv2.CV_64F).var())
        
        # Global motion (optical flow magnitude)
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0
            )
            motions.append(np.mean(np.sqrt(flow[...,0]**2 + flow[...,1]**2)))
        
        prev_gray = gray
        frame_count += 1
    
    cap.release()
    return {
        'mean_brightness': np.mean(brightnesses) if brightnesses else 0,
        'blur_variance': np.mean(blurs) if blurs else 0,
        'motion_magnitude': np.mean(motions) if motions else 0,
    }

# Compute proxies for test videos with errors
error_videos = [(video_names[i], video_max_scores[i], video_labels[i]) 
                for i in range(len(video_labels))
                if (video_labels[i] == 1 and video_max_scores[i] < threshold) or
                   (video_labels[i] == 0 and video_max_scores[i] >= threshold)]

print(f'\nMisclassified videos: {len(error_videos)}')
for name, score, label in error_videos[:10]:
    status = 'False Negative' if label == 1 else 'False Positive'
    print(f'  {status}: {name} (score={score:.4f})')

## 7. ROC & PR Curves

In [ ]:
curve_path = os.path.join(config.RUNS_DIR, f'roc_pr_curves_seed{config.SEED}.png')
plot_roc_pr_curves(
    np.array(all_clip_scores),
    np.array(all_clip_labels),
    save_path=curve_path
)
print(f'Curves saved to: {curve_path}')

In [ ]:
metrics = {
    'seed': config.SEED,
    'video_level_auc': video_auc,
    'clip_level_roc_auc': clip_metrics['roc_auc'],
    'clip_level_pr_auc': clip_metrics['pr_auc'],
    'threshold': threshold,
    'num_test_videos': len(video_labels),
    'num_normal_test': sum(1 for l in video_labels if l == 0),
    'num_anomalous_test': sum(1 for l in video_labels if l == 1),
}

metrics_path = os.path.join(config.RUNS_DIR, f'metrics_seed{config.SEED}.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nAll metrics saved to: {metrics_path}')
print(json.dumps(metrics, indent=2))

Deliverable summary: The `metrics_seed<SEED>.json` and the `calibrated_threshold.json` files contain the final evaluation metadata required to pass the test phase.